# MetroRide Phase 1 with PySpark and DuckLake

This tutorial builds the same bronze and silver layers as `../tutorial.ipynb`, but **all data processing is done with PySpark**. DuckLake still owns the tables, and **pyducklake** is used only to talk to the catalog.

| Who does what | Tool |
|---|---|
| Create namespaces and tables | pyducklake |
| Read CSV files, add metadata, find duplicates | **PySpark** |
| Parse types, derive columns, validate (bronze → silver) | **PySpark** |
| Show and query data | **PySpark** |
| Commit a Spark DataFrame to a DuckLake table, read a table into Spark | pyducklake ↔ PySpark, through Apache Arrow |
| Snapshot information | pyducklake |

```
 CSV ──Spark read──▶ Spark DataFrame ──toArrow()──▶ pyducklake append ──▶ bronze.trips_raw
                                                                               │
 silver.trips ◀── pyducklake append ◀──toArrow()── Spark transform ◀──Arrow── read
```

**Why Arrow?** There is no mature Spark connector for DuckLake yet. Arrow is a columnar in-memory format that both Spark and pyducklake understand, so a DataFrame can move between them in one call. Here the data passes through the Spark driver, which is fine for a classroom-sized dataset.

| Step | What you do |
|---|---|
| 1 | Create all tables (pyducklake) |
| 2 | Ingest April into bronze (PySpark) |
| 3 | Transform bronze → silver (PySpark) |
| 4 | Show both layers (PySpark) |
| 5 | Ingest May into bronze, append mode, no duplicate rows (PySpark) |
| 6 | Transform the new bronze rows → silver (PySpark) |
| 7 | Show both layers (PySpark) |
| 8 | Show all snapshot information |

**Requirements**

* Java 17 or newer (PySpark 4 runs on the JVM). Check with `java -version`.
* Python packages: `pip install "pyducklake[pandas]" pytz pyspark "pandas<3"`, or in the uv project: `uv add pyspark "pandas<3"`.

Run this notebook from the `phases/phase-1-batch/pyspark/` folder. It reads the CSVs from `../bronze/` and keeps its own lakehouse in `pyspark/lakehouse/`, separate from the first tutorial.

## 0. Setup

### 0.1 The DuckLake catalog

Same as in the first tutorial: a catalog database (`metadata.ducklake`) plus a folder of Parquet files (`data/`). The lakehouse folder is recreated on every run.

In [1]:
import shutil
from pathlib import Path

import pyarrow as pa
from pyducklake import (
    Catalog, Schema, required, optional,
    StringType, IntegerType, TimestampTZType, DecimalType, DateType,
)

TUTORIAL_DIR = Path.cwd()                      # .../phases/phase-1-batch/pyspark
SOURCE_DIR = TUTORIAL_DIR.parent / "bronze"    # nightly CSV exports
LAKE_DIR = TUTORIAL_DIR / "lakehouse"          # this tutorial's own lakehouse
assert SOURCE_DIR.exists(), "Run this notebook from the phases/phase-1-batch/pyspark/ folder"

if LAKE_DIR.exists():                          # start clean on every run
    shutil.rmtree(LAKE_DIR)
LAKE_DIR.mkdir()

catalog = Catalog("metroride", str(LAKE_DIR / "metadata.ducklake"), data_path=str(LAKE_DIR / "data"))
catalog.set_option("data_inlining_row_limit", "0")      # every insert becomes a Parquet file
catalog.connection.execute("SET TimeZone = 'Asia/Bangkok'")

print("Catalog:", (LAKE_DIR / "metadata.ducklake").relative_to(TUTORIAL_DIR))

Catalog: lakehouse/metadata.ducklake


### 0.2 The Spark session

A local Spark session that uses all CPU cores of your machine. Two settings matter:

* `spark.sql.session.timeZone = Asia/Bangkok`: Spark shows timestamps, and computes dates and hours, in Bangkok time.
* `spark.sql.execution.arrow.pyspark.enabled = true`: use Arrow to move data between Spark and Python.

Starting Spark takes a few seconds the first time.

In [2]:
from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pyspark.sql import types as T

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("metroride-phase1")
    .config("spark.sql.session.timeZone", "Asia/Bangkok")
    .config("spark.sql.execution.arrow.pyspark.enabled", "true")
    .config("spark.sql.shuffle.partitions", "4")     # small data: few partitions
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

Spark 4.2.0


### 0.3 The bridge between Spark and DuckLake

Two small functions are all we need:

* `to_spark(table)` reads a DuckLake table with pyducklake and turns it into a **Spark DataFrame**.
* `append_from_spark(table_id, df, message)` turns a Spark DataFrame into Arrow and **appends** it to a DuckLake table in one transaction with a commit message. A `{n}` in the message is replaced by the number of rows written.

`append_from_spark` first selects the table's columns **by name** and casts them to the table's exact Arrow types (for example `int32`, `decimal(10,2)`, timestamp with time zone). This matters because pyducklake's `append` matches columns by position.

In [3]:
def to_spark(table, snapshot_id=None):
    # Read a DuckLake table (optionally at an older snapshot) into a Spark DataFrame.
    scan = table.scan() if snapshot_id is None else table.scan().with_snapshot(snapshot_id)
    return spark.createDataFrame(scan.to_arrow())


def append_from_spark(table_id, df, message):
    # Append a Spark DataFrame to a DuckLake table in one commit. Returns the number of rows written.
    table = catalog.load_table(table_id)
    target = table.scan().with_limit(0).to_arrow().schema          # the table's exact column types
    rows = df.select(*target.names).toArrow().cast(target)         # Spark -> Arrow, by column name
    if rows.num_rows == 0:
        return 0
    with catalog.begin_transaction() as txn:
        catalog.set_commit_message(message.format(n=rows.num_rows), author="phase-1 pyspark tutorial")
        txn.load_table(table_id).append(rows)
    return rows.num_rows

## 1. Create all tables (pyducklake)

The same two tables as in the first tutorial:

* **`bronze.trips_raw`** has every CSV column as text, plus `source_file`, `source_row_number`, `file_sha256` and `ingested_at`.
* **`silver.trips`** has one validated, typed row per `trip_id`, plus the derived `service_date` and `time_slot`.

In [4]:
catalog.create_namespace("bronze")
catalog.create_namespace("silver")

TRIP_COLUMNS = [
    "trip_id", "driver_id", "service_zone", "requested_at", "accepted_at",
    "picked_up_at", "completed_at", "cancelled_at", "status", "fare_thb",
]

bronze = catalog.create_table(("bronze", "trips_raw"), Schema.of(
    *[optional(col, StringType()) for col in TRIP_COLUMNS],
    required("source_file", StringType()),
    required("source_row_number", IntegerType()),
    required("file_sha256", StringType()),
    required("ingested_at", TimestampTZType()),
))

silver = catalog.create_table(("silver", "trips"), Schema.of(
    required("trip_id", StringType()),
    required("driver_id", StringType()),
    required("service_zone", StringType()),
    required("status", StringType()),
    required("requested_at", TimestampTZType()),
    required("accepted_at", TimestampTZType()),
    optional("picked_up_at", TimestampTZType()),
    optional("completed_at", TimestampTZType()),
    optional("cancelled_at", TimestampTZType()),
    optional("fare_thb", DecimalType(10, 2)),
    required("service_date", DateType()),
    required("time_slot", StringType()),
))

for ns in ["bronze", "silver"]:
    print(ns, "->", catalog.list_tables(ns))

bronze -> [('bronze', 'trips_raw')]
silver -> [('silver', 'trips')]


Spark sees the empty tables with these Spark types:

In [5]:
to_spark(bronze).printSchema()
to_spark(silver).printSchema()

root
 |-- trip_id: string (nullable = true)
 |-- driver_id: string (nullable = true)
 |-- service_zone: string (nullable = true)
 |-- requested_at: string (nullable = true)
 |-- accepted_at: string (nullable = true)
 |-- picked_up_at: string (nullable = true)
 |-- completed_at: string (nullable = true)
 |-- cancelled_at: string (nullable = true)
 |-- status: string (nullable = true)
 |-- fare_thb: string (nullable = true)
 |-- source_file: string (nullable = true)
 |-- source_row_number: integer (nullable = true)
 |-- file_sha256: string (nullable = true)
 |-- ingested_at: timestamp (nullable = true)

root
 |-- trip_id: string (nullable = true)
 |-- driver_id: string (nullable = true)
 |-- service_zone: string (nullable = true)
 |-- status: string (nullable = true)
 |-- requested_at: timestamp (nullable = true)
 |-- accepted_at: timestamp (nullable = true)
 |-- picked_up_at: timestamp (nullable = true)
 |-- completed_at: timestamp (nullable = true)
 |-- cancelled_at: timestamp (nullabl

## 2. Ingest April into bronze (PySpark)

`ingest_trips_csv()` does everything in Spark:

1. **Fingerprint the file.** Spark reads it as one binary record (`binaryFile` format), and `sha2(content, 256)` gives its SHA-256 hash.
2. **File guard.** If that hash is already in bronze, skip the whole file.
3. **Read the CSV as text.** We give Spark a schema of `StringType` columns, so nothing is converted. Blank cells become `NULL`. `enforceSchema=False` makes Spark check that the CSV header matches our column names.
4. **Add metadata.** `source_row_number` is the row's position in the file: `monotonically_increasing_id()` keeps the file order and `row_number()` turns it into 1, 2, 3, …
5. **Row guard.** Keep the first copy of each `trip_id` inside the file (window + `row_number`), then drop every `trip_id` already in bronze with a **left anti join**.
6. **Append** the remaining rows with `append_from_spark()`.

In [6]:
CSV_SCHEMA = T.StructType([T.StructField(col, T.StringType()) for col in TRIP_COLUMNS])


def ingest_trips_csv(csv_path: Path) -> int:
    # Append one nightly CSV to bronze.trips_raw using Spark. Returns the number of rows inserted.
    path = str(csv_path)

    # 1. Fingerprint the whole file
    file_hash = (spark.read.format("binaryFile").load(path)
                 .select(F.sha2("content", 256).alias("sha256"))
                 .first()["sha256"])

    # 2. File guard: the exact same file is already in bronze
    bronze_now = to_spark(bronze)
    if bronze_now.filter(F.col("file_sha256") == file_hash).limit(1).count() > 0:
        print(f"SKIP {csv_path.name}: this file (sha256 {file_hash[:12]}...) is already in bronze")
        return 0

    # 3. Read every column as text
    raw = (spark.read
           .option("header", True)
           .option("enforceSchema", False)        # fail if the header does not match CSV_SCHEMA
           .schema(CSV_SCHEMA)
           .csv(path))

    # 4. Add ingest metadata
    file_order = Window.orderBy(F.monotonically_increasing_id())
    batch = (raw
             .withColumn("source_file", F.lit(csv_path.name))
             .withColumn("source_row_number", F.row_number().over(file_order))
             .withColumn("file_sha256", F.lit(file_hash))
             .withColumn("ingested_at", F.current_timestamp()))

    # 5. Row guard: first copy per trip_id, and only trip_ids not yet in bronze
    first_copy = Window.partitionBy("trip_id").orderBy("source_row_number")
    new_rows = (batch
                .withColumn("copy_no", F.row_number().over(first_copy))
                .filter(F.col("copy_no") == 1)
                .drop("copy_no")
                .join(bronze_now.select("trip_id"), on="trip_id", how="left_anti"))

    # 6. Append in one DuckLake commit
    rows_read = raw.count()
    appended = append_from_spark(("bronze", "trips_raw"), new_rows, f"bronze: ingest {csv_path.name} (pyspark)")
    if appended == 0:
        print(f"SKIP {csv_path.name}: all {rows_read} rows are duplicates of rows already in bronze")
    else:
        print(f"OK   {csv_path.name}: {rows_read} rows read, {appended} appended, {rows_read - appended} duplicates skipped")
    return appended

In [7]:
ingest_trips_csv(SOURCE_DIR / "trips_2026-04.csv")

OK   trips_2026-04.csv: 132 rows read, 132 appended, 0 duplicates skipped


132

## 3. Transform bronze → silver (PySpark)

`transform_bronze_to_silver()` uses the Spark DataFrame API in three stages:

1. **Pick new rows:** bronze rows whose `trip_id` is not in silver yet (**left anti join**). This makes the transform incremental and idempotent.
2. **Parse and derive:**
   * `try_to_timestamp` parses the `+07:00` timestamps;
   * `try_cast("decimal(10,2)")` parses the fare;
   * both return `NULL` instead of failing on a bad value;
   * `to_date` and `hour` derive `service_date` and `time_slot` in the session time zone, Bangkok.
3. **Validate:** keep only rows that pass every phase-1 rule, and one row per `trip_id`. Rows that fail stay in bronze.

| Status | Rule |
|---|---|
| all | ids present; zone in Siam/Asok/Ari/Silom; `requested_at <= accepted_at`; request at 08:xx or 18:xx |
| `completed` | `accepted_at <= picked_up_at <= completed_at`; fare present and `>= 0`; no `cancelled_at` |
| `cancelled` | `accepted_at <= cancelled_at`; no pickup, completion or fare |

In [8]:
TS_FORMAT = "yyyy-MM-dd'T'HH:mm:ssXXX"          # e.g. 2026-04-01T08:03:00+07:00
ZONES = ["Siam", "Asok", "Ari", "Silom"]
c = F.col


def parse_ts(column: str):
    return F.try_to_timestamp(c(column), F.lit(TS_FORMAT))


def transform_bronze_to_silver() -> int:
    # Move new, valid bronze rows into silver.trips. Returns the number of rows appended.

    # 1. Bronze rows that are not in silver yet
    new_rows = to_spark(bronze).join(to_spark(silver).select("trip_id"), on="trip_id", how="left_anti")

    # 2. Parse types and derive columns
    typed = (new_rows.select(
                F.trim("trip_id").alias("trip_id"),
                F.trim("driver_id").alias("driver_id"),
                F.trim("service_zone").alias("service_zone"),
                F.lower(F.trim("status")).alias("status"),
                parse_ts("requested_at").alias("requested_at"),
                parse_ts("accepted_at").alias("accepted_at"),
                parse_ts("picked_up_at").alias("picked_up_at"),
                parse_ts("completed_at").alias("completed_at"),
                parse_ts("cancelled_at").alias("cancelled_at"),
                c("fare_thb").try_cast("decimal(10,2)").alias("fare_thb"),
                "ingested_at",
             )
             .withColumn("service_date", F.to_date("requested_at"))      # Bangkok date (session time zone)
             .withColumn("time_slot", F.when(F.hour("requested_at") == 8, "morning")
                                       .when(F.hour("requested_at") == 18, "evening")))

    # 3. Validate
    valid_for_all = (
        c("trip_id").isNotNull() & c("driver_id").isNotNull()
        & c("service_zone").isin(ZONES)
        & c("requested_at").isNotNull() & c("accepted_at").isNotNull()
        & (c("requested_at") <= c("accepted_at"))
        & c("time_slot").isNotNull()
    )
    valid_completed = (
        (c("status") == "completed")
        & c("picked_up_at").isNotNull() & c("completed_at").isNotNull()
        & (c("accepted_at") <= c("picked_up_at")) & (c("picked_up_at") <= c("completed_at"))
        & c("fare_thb").isNotNull() & (c("fare_thb") >= 0)
        & c("cancelled_at").isNull()
    )
    valid_cancelled = (
        (c("status") == "cancelled")
        & c("cancelled_at").isNotNull() & (c("accepted_at") <= c("cancelled_at"))
        & c("picked_up_at").isNull() & c("completed_at").isNull() & c("fare_thb").isNull()
    )
    one_per_trip = Window.partitionBy("trip_id").orderBy("ingested_at")
    clean = (typed
             .filter(valid_for_all & (valid_completed | valid_cancelled))
             .withColumn("copy_no", F.row_number().over(one_per_trip))
             .filter(c("copy_no") == 1)
             .drop("copy_no", "ingested_at"))

    pending = new_rows.count()
    appended = append_from_spark(("silver", "trips"), clean, "silver: transform {n} new trips (pyspark)")
    if appended == 0:
        print(f"Nothing to do: {pending} bronze rows not yet in silver, 0 valid rows to append")
    else:
        print(f"{pending} bronze rows not yet in silver -> {appended} appended, {pending - appended} rejected by validation")
    return appended

In [9]:
transform_bronze_to_silver()

132 bronze rows not yet in silver -> 132 appended, 0 rejected by validation


132

## 4. Show both layers (PySpark)

We load each table into Spark and register it as a **temporary view**, so we can use both the DataFrame API and Spark SQL. `show_layers()` is reused in Step 7.

In [10]:
def show_layers():
    # Load both DuckLake tables into Spark and register them as temporary views.
    bronze_df = to_spark(bronze)
    silver_df = to_spark(silver)
    bronze_df.createOrReplaceTempView("bronze_trips_raw")
    silver_df.createOrReplaceTempView("silver_trips")
    print(f"bronze.trips_raw: {bronze_df.count()} rows | silver.trips: {silver_df.count()} rows")
    return bronze_df, silver_df


bronze_df, silver_df = show_layers()

bronze.trips_raw: 132 rows | silver.trips: 132 rows


**Bronze**: every source value is still text, and the ingest metadata is attached:

In [11]:
(bronze_df
 .select("trip_id", "status", "requested_at", "picked_up_at", "fare_thb", "source_file", "source_row_number", "ingested_at")
 .orderBy("source_row_number")
 .show(5, truncate=False))

+---------------+---------+-------------------------+-------------------------+--------+-----------------+-----------------+--------------------------+
|trip_id        |status   |requested_at             |picked_up_at             |fare_thb|source_file      |source_row_number|ingested_at               |
+---------------+---------+-------------------------+-------------------------+--------+-----------------+-----------------+--------------------------+
|BKK-20260401-M1|completed|2026-04-01T08:03:00+07:00|2026-04-01T08:11:00+07:00|148.00  |trips_2026-04.csv|1                |2026-10-18 20:54:19.684993|
|BKK-20260401-M2|completed|2026-04-01T08:20:00+07:00|2026-04-01T08:31:00+07:00|161.00  |trips_2026-04.csv|2                |2026-10-18 20:54:19.684993|
|BKK-20260401-E1|completed|2026-04-01T18:03:00+07:00|2026-04-01T18:15:00+07:00|166.00  |trips_2026-04.csv|3                |2026-10-18 20:54:19.684993|
|BKK-20260401-E2|completed|2026-04-01T18:20:00+07:00|2026-04-01T18:35:00+07:00|179.00  |

**Silver**: real types and derived columns:

In [12]:
silver_df.printSchema()
(silver_df
 .select("trip_id", "status", "requested_at", "picked_up_at", "fare_thb", "service_date", "time_slot")
 .orderBy("requested_at")
 .show(5, truncate=False))

root
 |-- trip_id: string (nullable = true)
 |-- driver_id: string (nullable = true)
 |-- service_zone: string (nullable = true)
 |-- status: string (nullable = true)
 |-- requested_at: timestamp (nullable = true)
 |-- accepted_at: timestamp (nullable = true)
 |-- picked_up_at: timestamp (nullable = true)
 |-- completed_at: timestamp (nullable = true)
 |-- cancelled_at: timestamp (nullable = true)
 |-- fare_thb: decimal(10,2) (nullable = true)
 |-- service_date: date (nullable = true)
 |-- time_slot: string (nullable = true)

+---------------+---------+-------------------+-------------------+--------+------------+---------+
|trip_id        |status   |requested_at       |picked_up_at       |fare_thb|service_date|time_slot|
+---------------+---------+-------------------+-------------------+--------+------------+---------+
|BKK-20260401-M1|completed|2026-04-01 08:03:00|2026-04-01 08:11:00|148.00  |2026-04-01  |morning  |
|BKK-20260401-M2|completed|2026-04-01 08:20:00|2026-04-01 08:31:00|1

The same data with Spark SQL: requests, completed trips and fares per time slot.

In [13]:
spark.sql('''
    SELECT time_slot,
           count(*)                                        AS requests,
           count_if(status = 'completed')                  AS completed,
           count_if(status = 'cancelled')                  AS cancelled,
           sum(fare_thb)                                   AS gross_fare_thb
    FROM silver_trips
    GROUP BY time_slot
    ORDER BY time_slot DESC
''').show()

+---------+--------+---------+---------+--------------+
|time_slot|requests|completed|cancelled|gross_fare_thb|
+---------+--------+---------+---------+--------------+
|  morning|      66|       61|        5|      11161.00|
|  evening|      66|       63|        3|      11414.00|
+---------+--------+---------+---------+--------------+



## 5. Ingest May into bronze (append mode, no duplicate rows)

The normal nightly load of May:

In [14]:
ingest_trips_csv(SOURCE_DIR / "trips_2026-05.csv")

OK   trips_2026-05.csv: 137 rows read, 137 appended, 0 duplicates skipped


137

Now we test the duplicate guards.

**Replay:** the job is re-run over every file. Both hashes are already in bronze, so the **file guard** skips them.

In [15]:
for csv_file in sorted(SOURCE_DIR.glob("trips_*.csv")):
    ingest_trips_csv(csv_file)

SKIP trips_2026-04.csv: this file (sha256 ee6286bcdaff...) is already in bronze
SKIP trips_2026-05.csv: this file (sha256 d9b4c1ca04b9...) is already in bronze


**Re-sent export:** a *different* file that repeats three May trips. Its hash is new, so it passes the file guard, but the **row guard** (left anti join) finds every `trip_id` already in bronze.

In [16]:
resend_dir = LAKE_DIR / "landing"
resend_dir.mkdir(exist_ok=True)
may_lines = (SOURCE_DIR / "trips_2026-05.csv").read_text().splitlines()
resend_file = resend_dir / "trips_2026-05_resend.csv"
resend_file.write_text("\n".join(may_lines[:4]) + "\n")       # header + first 3 May trips

ingest_trips_csv(resend_file)

SKIP trips_2026-05_resend.csv: all 3 rows are duplicates of rows already in bronze


0

In [17]:
to_spark(bronze).agg(
    F.count("*").alias("bronze_rows"),
    F.countDistinct("trip_id").alias("distinct_trip_ids"),
).withColumn("duplicate_rows", c("bronze_rows") - c("distinct_trip_ids")).show()

+-----------+-----------------+--------------+
|bronze_rows|distinct_trip_ids|duplicate_rows|
+-----------+-----------------+--------------+
|        269|              269|             0|
+-----------+-----------------+--------------+



## 6. Transform the new bronze rows → silver (PySpark)

The same function as Step 3. The left anti join finds only the 137 May trips. Running it again does nothing.

In [18]:
transform_bronze_to_silver()

137 bronze rows not yet in silver -> 137 appended, 0 rejected by validation


137

In [19]:
transform_bronze_to_silver()     # idempotent: nothing new to insert

Nothing to do: 0 bronze rows not yet in silver, 0 valid rows to append


0

## 7. Show both layers again (PySpark)

In [20]:
bronze_df, silver_df = show_layers()

bronze.trips_raw: 269 rows | silver.trips: 269 rows


Rows per month in each layer, with Spark SQL over the two temporary views:

In [21]:
spark.sql('''
    SELECT 'bronze.trips_raw' AS table_name, substr(requested_at, 1, 7) AS month, count(*) AS rows
    FROM bronze_trips_raw GROUP BY 2
    UNION ALL
    SELECT 'silver.trips', date_format(service_date, 'yyyy-MM'), count(*)
    FROM silver_trips GROUP BY 2
    ORDER BY table_name, month
''').show()

+----------------+-------+----+
|      table_name|  month|rows|
+----------------+-------+----+
|bronze.trips_raw|2026-04| 132|
|bronze.trips_raw|2026-05| 137|
|    silver.trips|2026-04| 132|
|    silver.trips|2026-05| 137|
+----------------+-------+----+



Ingestion log per source file, from bronze's metadata columns:

In [22]:
(bronze_df
 .groupBy("source_file")
 .agg(F.count("*").alias("rows"),
      F.min("source_row_number").alias("first_row"),
      F.max("source_row_number").alias("last_row"),
      F.min("ingested_at").alias("ingested_at"),
      F.substring(F.first("file_sha256"), 1, 16).alias("file_sha256_prefix"))
 .orderBy("ingested_at")
 .show(truncate=False))

+-----------------+----+---------+--------+--------------------------+------------------+
|source_file      |rows|first_row|last_row|ingested_at               |file_sha256_prefix|
+-----------------+----+---------+--------+--------------------------+------------------+
|trips_2026-04.csv|132 |1        |132     |2026-10-18 20:54:19.684993|ee6286bcdafff7d1  |
|trips_2026-05.csv|137 |1        |137     |2026-10-18 20:54:26.026007|d9b4c1ca04b94aa9  |
+-----------------+----+---------+--------+--------------------------+------------------+



The same May trip in both layers: text in bronze, typed and enriched in silver.

In [23]:
bronze_df.filter(c("trip_id") == "BKK-20260501-M1").show(vertical=True, truncate=False)
silver_df.filter(c("trip_id") == "BKK-20260501-M1").show(vertical=True, truncate=False)

-RECORD 0-----------------------------------------------------------------------------
 trip_id           | BKK-20260501-M1                                                  
 driver_id         | D002                                                             
 service_zone      | Asok                                                             
 requested_at      | 2026-05-01T08:03:00+07:00                                        
 accepted_at       | 2026-05-01T08:05:00+07:00                                        
 picked_up_at      | 2026-05-01T08:11:00+07:00                                        
 completed_at      | 2026-05-01T08:24:00+07:00                                        
 cancelled_at      | NULL                                                             
 status            | completed                                                        
 fare_thb          | 148.00                                                           
 source_file       | trips_2026-05.csv     

May figures per time slot in silver:

In [24]:
(silver_df
 .filter(F.date_format("service_date", "yyyy-MM") == "2026-05")
 .groupBy("time_slot")
 .agg(F.count("*").alias("requests"),
      F.count_if(c("status") == "completed").alias("completed"),
      F.sum("fare_thb").alias("gross_fare_thb"))
 .orderBy(F.desc("time_slot"))
 .show())

+---------+--------+---------+--------------+
|time_slot|requests|completed|gross_fare_thb|
+---------+--------+---------+--------------+
|  morning|      68|       62|      11366.00|
|  evening|      69|       65|      11723.00|
+---------+--------+---------+--------------+



## 8. Show all snapshot information

Every commit made by `append_from_spark` created a DuckLake snapshot with our commit message. Re-runs that wrote nothing (the replay, the re-sent file, the second transform) created **no** snapshot.

In [25]:
import pandas as pd
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 60)

snapshots = bronze.inspect().snapshots().to_pandas()
snapshots[["snapshot_id", "snapshot_time", "schema_version", "changes", "author", "commit_message"]]

,snapshot_id,snapshot_time,schema_version,changes,author,commit_message
0,0,2026-10-18 20:54:09.585209+07:00,0,"[(schemas_created, [main])]",None,None
1,1,2026-10-18 20:54:15.628930+07:00,1,"[(schemas_created, [bronze])]",None,None
2,2,2026-10-18 20:54:15.654766+07:00,2,"[(schemas_created, [silver])]",None,None
3,3,2026-10-18 20:54:15.681710+07:00,3,"[(tables_created, [bronze.trips_raw])]",None,None
4,4,2026-10-18 20:54:15.738528+07:00,4,"[(tables_created, [silver.trips])]",None,None
5,5,2026-10-18 20:54:20.776722+07:00,4,"[(tables_inserted_into, [3])]",phase-1 pyspark tutorial,bronze: ingest trips_2026-04.csv (pyspark)
6,6,2026-10-18 20:54:24.200068+07:00,4,"[(tables_inserted_into, [4])]",phase-1 pyspark tutorial,silver: transform 132 new trips (pyspark)
7,7,2026-10-18 20:54:26.504706+07:00,4,"[(tables_inserted_into, [3])]",phase-1 pyspark tutorial,bronze: ingest trips_2026-05.csv (pyspark)
8,8,2026-10-18 20:54:28.944558+07:00,4,"[(tables_inserted_into, [4])]",phase-1 pyspark tutorial,silver: transform 137 new trips (pyspark)


Snapshots are catalog-wide. To see the history of each table, join the catalog's metadata tables: which snapshot added which Parquet file, with how many rows.

In [26]:
catalog.connection.sql('''
    SELECT s.schema_name || '.' || t.table_name AS table_name,
           f.begin_snapshot                     AS snapshot_id,
           f.record_count                       AS rows_added,
           c.commit_message,
           f.path                               AS data_file
    FROM __ducklake_metadata_metroride.ducklake_data_file AS f
    JOIN __ducklake_metadata_metroride.ducklake_table  AS t USING (table_id)
    JOIN __ducklake_metadata_metroride.ducklake_schema AS s USING (schema_id)
    LEFT JOIN __ducklake_metadata_metroride.ducklake_snapshot_changes AS c ON c.snapshot_id = f.begin_snapshot
    ORDER BY table_name, snapshot_id
''').df()

,table_name,snapshot_id,rows_added,commit_message,data_file
0,bronze.trips_raw,5,132,bronze: ingest trips_2026-04.csv (pyspark),ducklake-01a14f4a-be0e-7cfa-a05c-2d199e59eff0.parquet
1,bronze.trips_raw,7,137,bronze: ingest trips_2026-05.csv (pyspark),ducklake-01a14f4a-d45b-75db-9e9a-b52debc34f57.parquet
2,silver.trips,6,132,silver: transform 132 new trips (pyspark),ducklake-01a14f4a-cb61-740d-a2f4-bc427421b9d2.parquet
3,silver.trips,8,137,silver: transform 137 new trips (pyspark),ducklake-01a14f4a-dde6-78da-94f6-e97344695ed0.parquet


Because every snapshot is kept, Spark can also read an **older version** of a table. Here is silver as it was right after the April transform:

In [27]:
silver_snapshots = snapshots[snapshots.commit_message.fillna("").str.startswith("silver:")].snapshot_id.tolist()
april_snapshot, may_snapshot = silver_snapshots[0], silver_snapshots[-1]

for label, sid in [("after April", april_snapshot), ("after May", may_snapshot)]:
    print(f"silver.trips at snapshot {sid} ({label}):", to_spark(silver, snapshot_id=sid).count(), "rows")

silver.trips at snapshot 6 (after April): 132 rows


silver.trips at snapshot 8 (after May): 269 rows


In [28]:
spark.stop()

## Summary

* **pyducklake** owns the catalog: it creates tables, commits appends as snapshots, and reports snapshot history.
* **PySpark** does all the data work: reading CSVs, hashing files, removing duplicates with window functions and left anti joins, parsing and validating with `try_to_timestamp` and `try_cast`, and showing results with the DataFrame API and Spark SQL.
* **Arrow** connects the two. `to_spark()` and `append_from_spark()` are the only places data crosses between them.

**Try it yourself**

1. Add one bad row to a copy of the April file (for example a completed trip without a fare) and check that it reaches bronze but not silver.
2. Rewrite the validation in Step 3 as one Spark SQL query on a temporary view and compare the results.
3. Build `gold.daily_service_performance` from `silver.trips` with Spark and commit it with `append_from_spark()`.